In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# 🚀 Section 3: Enter the KV Cache — The Hero We Needed

*Part of the Vizuara series on Efficient LLM Inference*
*Estimated time: 20–30 minutes*

## 1. Why Does This Matter?

Welcome back! In the previous sections we saw that a naive transformer inference loop is brutally expensive: every time we generate a new token, we recompute attention over the **entire** context from scratch. For a 2,000-token document, that means 2,000 full re-reads, every single step.

Today we meet the fix: the **KV Cache**. It is one of those ideas that feels almost too simple once you see it — and yet it is the single biggest reason that large language models are usable in production today.

Here is a teaser of what we will build and measure:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import time

torch.manual_seed(42)
np.random.seed(42)

print("✅ Imports ready. Let us build the KV Cache from first principles.")
print(f"   PyTorch version : {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"   Running on      : {device}")

By the end of this notebook you will have:

1. Implemented **naive autoregressive attention** (the expensive baseline).
2. Implemented **KV-cached attention** (the hero).
3. **Measured and visualised** the wall-clock speedup across sequence lengths.
4. Built an intuition for *why* the cache helps — and what its hidden cost is.

## 2. Building Intuition

Let us think carefully before writing a single line of code.

### The Broken Photocopier Analogy

Imagine you are a legal clerk who must read a 1,000-page contract and then type a one-sentence summary of what comes next. You do this sentence by sentence.

A naive clerk — let us call them **Naive Nadia** — reads the *entire* contract from page 1 every single time they need to write the next sentence. Write sentence 1: re-read all 1,000 pages. Write sentence 2: re-read all 1,000 pages again. By the time she has written 100 sentences she has read 100,000 pages of the same contract.

A smarter clerk — **Cache-Aware Carlos** — realises something: *the contract has not changed*. What changed is only the most recent sentence he just wrote. So he keeps a sticky-note summary of every page he has already read and simply reads the new sentence before adding it to his notes. Each step costs only a tiny increment of work.

**Naive Nadia's work** grows as $O(n^2)$. **Cache-Aware Carlos's work** per step is $O(n)$ — he only processes the *new* token against accumulated notes.

### What Actually Changes Between Steps?

When we move from generating token $t$ to generating token $t+1$, let us ask a precise question: *which quantities in the attention calculation actually change?*

Recall that for each attention head, every token $i$ produces three vectors:

- **Query** $q_i$: "What am I looking for?"
- **Key** $k_i$: "What information do I advertise?"
- **Value** $v_i$: "What information do I carry?"

The key insight is that $k_i$ and $v_i$ for token $i$ are computed **solely from token $i$'s embedding** (plus the model weights). They do not depend on what token $i+1$, $i+2$, … are. So once we compute $k_i$ and $v_i$, they are **permanent** — they never need to change.

Only the **query** changes at each step: the new token produces a new $q_{t+1}$, which attends over all accumulated keys and values.

### 🤔 Think About This

> *If you were designing this system from scratch, knowing that keys and values never change once computed — what data structure would you use to avoid recomputing them?*

Take a moment to think. The answer is exactly what gives the KV Cache its name: a simple **cache** (a dictionary / buffer) that stores $K$ and $V$ matrices and grows by one row per generated token.

## 3. The Mathematics

Let us make this precise.

### Standard Self-Attention (One Head)

For a sequence of $n$ tokens with embedding dimension $d$, we project each token's representation $x_i \in \mathbb{R}^d$ into queries, keys, and values using learned weight matrices $W_Q, W_K, W_V \in \mathbb{R}^{d \times d_h}$:

$$q_i = x_i W_Q, \quad k_i = x_i W_K, \quad v_i = x_i W_V$$

We then stack all queries into $Q \in \mathbb{R}^{n \times d_h}$, and similarly for $K$ and $V$. The attention output is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_h}}\right) V$$

**What this equation says, computationally:**
- $QK^\top$ is an $n \times n$ matrix — every token asking "how relevant is every other token to me?" That is $O(n^2 d_h)$ multiply-adds.
- The softmax turns raw scores into a probability distribution over positions.
- Multiplying by $V$ mixes the value vectors according to those probabilities.

For naive autoregressive generation, we run this **full** $n \times n$ computation at every single decoding step.

### KV-Cached Attention (The Hero)

At decoding step $t$, the *only* new token is $x_t$. We compute:

$$q_t = x_t W_Q, \quad k_t = x_t W_K, \quad v_t = x_t W_V$$

We append $k_t$ and $v_t$ to our cache:

$$K_{\text{cache}} \leftarrow \begin{bmatrix} K_{\text{cache}} \\ k_t \end{bmatrix}, \quad V_{\text{cache}} \leftarrow \begin{bmatrix} V_{\text{cache}} \\ v_t \end{bmatrix}$$

Then the attention output for *this single new token* is just:

$$o_t = \text{softmax}\!\left(\frac{q_t \, K_{\text{cache}}^\top}{\sqrt{d_h}}\right) V_{\text{cache}}$$

**What this equation says, computationally:**
- $q_t$ is a **single row** vector of shape $(1, d_h)$.
- $q_t K_{\text{cache}}^\top$ produces a **single row** of attention scores of shape $(1, t)$ — one score per past token.
- We multiply by $V_{\text{cache}} \in \mathbb{R}^{t \times d_h}$ to get a single output vector.

The matrix product is now $O(t \cdot d_h)$ rather than $O(t^2 \cdot d_h)$. **We have eliminated one entire factor of $t$** per step.

### Cost Comparison

| Method | FLOPs per decoding step | Total FLOPs for $n$ tokens |
|---|---|---|
| Naive | $O(t^2 \cdot d_h)$ | $O(n^3 \cdot d_h)$ |
| KV Cache | $O(t \cdot d_h)$ | $O(n^2 \cdot d_h)$ |

That is a full order of magnitude reduction in asymptotic complexity. For $n = 2000$ tokens, that is potentially a **2000×** reduction in attention FLOPs.

## 4. Let's Build It — Component by Component

### 4.1 The Projection Layer

Every attention head starts by projecting token embeddings into query, key, and value spaces. Let us build this cleanly so we can reuse it in both implementations.

In [ ]:
class SingleHeadAttentionProjections(nn.Module):
    """
    Computes Q, K, V projections for a single attention head.

    Parameters
    ----------
    d_model : int  — dimension of the input token embedding
    d_head  : int  — dimension of each head's Q/K/V space
    """
    def __init__(self, d_model: int, d_head: int):
        super().__init__()
        self.W_Q = nn.Linear(d_model, d_head, bias=False)
        self.W_K = nn.Linear(d_model, d_head, bias=False)
        self.W_V = nn.Linear(d_model, d_head, bias=False)
        self.scale = d_head ** -0.5   # the 1/sqrt(d_h) scaling factor

    def forward(self, x):
        """
        x : Tensor of shape (batch, seq_len, d_model)
        Returns Q, K, V each of shape (batch, seq_len, d_head)
        """
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)
        return Q, K, V


# Quick sanity check
d_model, d_head = 64, 16
proj = SingleHeadAttentionProjections(d_model, d_head).to(device)

dummy_input = torch.randn(1, 10, d_model, device=device)   # batch=1, seq=10
Q, K, V = proj(dummy_input)

print(f"Input  shape : {dummy_input.shape}")
print(f"Q      shape : {Q.shape}   ✅ (batch, seq_len, d_head)")
print(f"K      shape : {K.shape}   ✅")
print(f"V      shape : {V.shape}   ✅")

### 4.2 The Attention Score Computation

Now let us implement the core attention formula. We will keep this as a standalone function so we can call it from both implementations.

In [ ]:
def compute_attention(Q, K, V, scale, causal_mask=True):
    """
    Standard scaled dot-product attention.

    Q : (batch, q_len, d_head)
    K : (batch, k_len, d_head)
    V : (batch, k_len, d_head)

    Returns output of shape (batch, q_len, d_head)
    and attention weights of shape (batch, q_len, k_len).
    """
    # Step 1: raw scores — (batch, q_len, k_len)
    scores = torch.bmm(Q, K.transpose(1, 2)) * scale

    # Step 2: causal masking (upper triangle → -inf so softmax → 0)
    if causal_mask and Q.shape[1] > 1:
        q_len, k_len = Q.shape[1], K.shape[1]
        mask = torch.triu(
            torch.ones(q_len, k_len, device=Q.device), diagonal=1
        ).bool()
        scores = scores.masked_fill(mask, float("-inf"))

    # Step 3: softmax over key dimension
    weights = F.softmax(scores, dim=-1)

    # Step 4: weighted sum of values
    output = torch.bmm(weights, V)
    return output, weights


# 📊 Visualisation: let us look at what an attention weight matrix looks like
torch.manual_seed(0)
seq_len = 8
d_h = 16
scale = d_h ** -0.5

Q_demo = torch.randn(1, seq_len, d_h)
K_demo = torch.randn(1, seq_len, d_h)
V_demo = torch.randn(1, seq_len, d_h)

_, weights_demo = compute_attention(Q_demo, K_demo, V_demo, scale, causal_mask=True)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights_demo[0].detach().numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_title("Causal Attention Weights\n(upper triangle is masked to zero)", fontsize=11)
ax.set_xlabel("Key position (token being attended to)")
ax.set_ylabel("Query position (token doing the attending)")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig("attention_weights.png", dpi=120)
plt.show()
print("📊 Notice the strictly lower-triangular structure — each token only attends to past tokens.")

### 4.3 Naive Autoregressive Inference (The Villain)

Now let us build the naive approach: for each new token we generate, we run **full** attention over the entire sequence so far.

In [ ]:
class NaiveAutoregressiveAttention(nn.Module):
    """
    Naive autoregressive attention.
    At every decoding step t, recomputes Q, K, V for ALL t tokens
    and runs full O(t^2) attention.
    """
    def __init__(self, d_model: int, d_head: int):
        super().__init__()
        self.proj = SingleHeadAttentionProjections(d_model, d_head)

    def generate_step_by_step(self, context_embeddings):
        """
        Simulate token-by-token generation.

        context_embeddings : (1, context_len, d_model)

        Returns:
            outputs   : list of output vectors, one per decoding position
            time_per_step : list of wall-clock times (seconds)
        """
        outputs = []
        time_per_step = []

        context_len = context_embeddings.shape[1]

        for t in range(1, context_len + 1):
            # Take only the first t tokens — simulating "we have generated t tokens so far"
            current_seq = context_embeddings[:, :t, :]   # (1, t, d_model)

            start = time.perf_counter()

            # Recompute EVERYTHING from scratch — every single step
            Q, K, V = self.proj(current_seq)
            out, _ = compute_attention(Q, K, V, self.proj.scale, causal_mask=True)
            # We only care about the output for the *last* token
            last_token_output = out[:, -1:, :]

            elapsed = time.perf_counter() - start

            outputs.append(last_token_output.detach())
            time_per_step.append(elapsed)

        return outputs, time_per_step


# Instantiate and do a quick forward-pass check
d_model, d_head = 128, 32
naive_model = NaiveAutoregressiveAttention(d_model, d_head).to(device)

test_embeddings = torch.randn(1, 5, d_model, device=device)
out_list, times_list = naive_model.generate_step_by_step(test_embeddings)

print(f"Naive model output per step: {len(out_list)} steps")
print(f"Each output shape           : {out_list[0].shape}  (batch=1, seq=1, d_head={d_head})")
print(f"First-step time             : {times_list[0]*1000:.3f} ms")

### 4.4 KV-Cached Inference (The Hero)

Now the elegant version. We maintain a running cache of keys and values, appending one row per step.

In [ ]:
class KVCachedAttention(nn.Module):
    """
    KV-cached autoregressive attention.

    At each decoding step t:
      - Compute Q, K, V for the SINGLE new token only.
      - Append K and V to the cache.
      - Run attention with a (1, d_head) query against the full cache.

    This is O(t) per step instead of O(t^2).
    """
    def __init__(self, d_model: int, d_head: int):
        super().__init__()
        self.proj = SingleHeadAttentionProjections(d_model, d_head)

    def generate_step_by_step(self, context_embeddings):
        """
        context_embeddings : (1, context_len, d_model)

        Returns:
            outputs       : list of output vectors, one per decoding position
            time_per_step : list of wall-clock times (seconds)
            cache_sizes   : list of ints — how many rows the cache had each step
        """
        outputs = []
        time_per_step = []
        cache_sizes = []

        # Initialise empty caches
        # Shape will grow to (1, t, d_head) as we generate
        K_cache = torch.zeros(1, 0, self.proj.W_K.out_features, device=context_embeddings.device)
        V_cache = torch.zeros(1, 0, self.proj.W_V.out_features, device=context_embeddings.device)

        context_len = context_embeddings.shape[1]

        for t in range(context_len):
            # Current token embedding — shape (1, 1, d_model)
            current_token = context_embeddings[:, t:t+1, :]

            start = time.perf_counter()

            # Only compute projections for THIS ONE TOKEN
            q_t, k_t, v_t = self.proj(current_token)   # each: (1, 1, d_head)

            # Grow the cache by one row
            K_cache = torch.cat([K_cache, k_t], dim=1)   # (1, t+1, d_head)
            V_cache = torch.cat([V_cache, v_t], dim=1)   # (1, t+1, d_head)

            # Attention: single query against all cached keys/values
            # No causal mask needed — q_t can attend to all past + current by design
            out, _ = compute_attention(q_t, K_cache, V_cache,
                                       self.proj.scale, causal_mask=False)

            elapsed = time.perf_counter() - start

            outputs.append(out.detach())
            time_per_step.append(elapsed)
            cache_sizes.append(K_cache.shape[1])

        return outputs, time_per_step, cache_sizes


# Sanity check — make sure KV cache and naive produce the same outputs
kv_model = KVCachedAttention(d_model, d_head).to(device)
# Share weights so outputs should match
kv_model.proj.load_state_dict(naive_model.proj.state_dict())

test_embeddings = torch.randn(1, 5, d_model, device=device)

naive_outs, _ = naive_model.generate_step_by_step(test_embeddings)
kv_outs, _, cache_sz = kv_model.generate_step_by_step(test_embeddings)

print("✅ Verifying that KV cache matches naive output (they must agree!):")
for i, (n_out, kv_out) in enumerate(zip(naive_outs, kv_outs)):
    diff = (n_out - kv_out).abs().max().item()
    match = "✅" if diff < 1e-5 else "❌"
    print(f"   Step {i+1}: max abs diff = {diff:.2e}  {match}")

## 5. 🔧 Your Turn

Now it is your turn to practise. We want you to write a function that **benchmarks** both implementations across a range of sequence lengths and returns the timing data. This is the function that will power our big visualisation.

Do not worry — we have scaffolded every step for you.

In [ ]:
def benchmark_attention_methods(seq_lengths, d_model=128, d_head=32, num_warmup=2):
    """
    Benchmark naive vs KV-cached attention across sequence lengths.

    Parameters
    ----------
    seq_lengths : list of int — sequence lengths to benchmark
    d_model     : int         — token embedding dimension
    d_head      : int         — attention head dimension
    num_warmup  : int         — number of warmup runs to discard (for timing stability)

    Returns
    -------
    results : dict with keys:
        "seq_lengths"       : list of int
        "naive_total_ms"    : list of float  — total wall-clock time (ms) for naive
        "kv_total_ms"       : list of float  — total wall-clock time (ms) for KV cache
        "speedup"           : list of float  — naive / kv for each length
    """
    # ============ TODO ============
    # Step 1: Create one NaiveAutoregressiveAttention and one KVCachedAttention,
    #         both with the given d_model and d_head. Move them to `device`.
    #         Share weights (call kv_model.proj.load_state_dict(naive_model.proj.state_dict()))
    #         so the outputs are comparable.

    # Step 2: For each seq_len in seq_lengths:
    #   a. Create a random context_embeddings tensor of shape (1, seq_len, d_model)
    #      on `device`, using torch.randn. Set torch.manual_seed(seq_len) for reproducibility.
    #   b. Run `num_warmup` rounds of each method and DISCARD those timings (they include
    #      JIT compilation overhead). Then run ONE real timed benchmark for each method
    #      using generate_step_by_step.
    #   c. Compute total_ms = sum(time_per_step) * 1000 for each method.
    #   d. Compute speedup = naive_total_ms / kv_total_ms.
    #   e. Append results.

    # Step 3: Return the results dict with keys listed in the docstring.

    # ============ END TODO ========

    results = {
        "seq_lengths"    : [],
        "naive_total_ms" : [],
        "kv_total_ms"    : [],
        "speedup"        : [],
    }

    # YOUR CODE HERE — replace the line below
    raise NotImplementedError("Complete the TODO above! 💪")

    return results

Once you have implemented it, run the verification cell below:

In [ ]:
# ✅ Verification cell — run this after completing your TODO

try:
    _test_results = benchmark_attention_methods(
        seq_lengths=[10, 20, 30],
        d_model=64, d_head=16
    )
    assert "seq_lengths"    in _test_results, "Missing key: seq_lengths"
    assert "naive_total_ms" in _test_results, "Missing key: naive_total_ms"
    assert "kv_total_ms"    in _test_results, "Missing key: kv_total_ms"
    assert "speedup"        in _test_results, "Missing key: speedup"
    assert len(_test_results["seq_lengths"]) == 3, "Should have 3 results"
    assert all(s > 0 for s in _test_results["speedup"]), "Speedups should be positive"
    assert all(s >= 1.0 for s in _test_results["speedup"]), \
        "KV cache should be at least as fast as naive (speedup >= 1)"
    print("✅ All checks passed! Your benchmark function works correctly.")
    print(f"   Speedups measured: {[round(s,2) for s in _test_results['speedup']]}")
except NotImplementedError:
    print("⚠️  You still need to implement the TODO. Give it a go — you have all the pieces!")
except AssertionError as e:
    print(f"❌ Check failed: {e}\n   Look at the failing assertion and revisit that step.")

> 💡 **Hint if you are stuck:** The warmup loop is just a `for _ in range(num_warmup):` block where you call `generate_step_by_step` but throw away the return value. The real timing run is the same call, but you keep `time_per_step`.

## 6. Putting It All Together

Let us now run the full benchmark and see the speedup in action. We provide a complete reference implementation here so the notebook can run end-to-end even if the TODO section is still a work in progress.

In [ ]:
# Reference implementation — study this after trying your own solution!

def benchmark_attention_methods_reference(seq_lengths, d_model=128, d_head=32, num_warmup=2):
    """Reference implementation of the benchmark function."""
    naive_model_ref = NaiveAutoregressiveAttention(d_model, d_head).to(device)
    kv_model_ref    = KVCachedAttention(d_model, d_head).to(device)
    kv_model_ref.proj.load_state_dict(naive_model_ref.proj.state_dict())

    results = {
        "seq_lengths"    : [],
        "naive_total_ms" : [],
        "kv_total_ms"    : [],
        "speedup"        : [],
    }

    for seq_len in seq_lengths:
        torch.manual_seed(seq_len)
        context_emb = torch.randn(1, seq_len, d_model, device=device)

        # --- Warmup ---
        for _ in range(num_warmup):
            naive_model_ref.generate_step_by_step(context_emb)
            kv_model_ref.generate_step_by_step(context_emb)

        # --- Real benchmark ---
        _, naive_times  = naive_model_ref.generate_step_by_step(context_emb)
        _, kv_times, _  = kv_model_ref.generate_step_by_step(context_emb)

        naive_ms = sum(naive_times) * 1000
        kv_ms    = sum(kv_times) * 1000
        speedup  = naive_ms / kv_ms if kv_ms > 0 else float("inf")

        results["seq_lengths"].append(seq_len)
        results["naive_total_ms"].append(naive_ms)
        results["kv_total_ms"].append(kv_ms)
        results["speedup"].append(speedup)

        print(f"   seq_len={seq_len:4d} | "
              f"Naive: {naive_ms:7.2f} ms | "
              f"KV Cache: {kv_ms:7.2f} ms | "
              f"Speedup: {speedup:.2f}×")

    return results


# Run the benchmark
seq_lengths_to_test = [50, 100, 200, 300, 500, 750, 1000]
print("🔬 Running benchmark — this may take a minute...\n")
results = benchmark_attention_methods_reference(
    seq_lengths=seq_lengths_to_test, d_model=128, d_head=32
)
print("\n✅ Benchmark complete!")

## 7. Analysing the Results

Now let us analyse what we measured with a detailed visualisation.

In [ ]:
# 📊 Main results figure: three-panel analysis

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("KV Cache vs Naive Attention — Benchmark Results", fontsize=14, fontweight="bold")

seq_arr      = np.array(results["seq_lengths"])
naive_arr    = np.array(results["naive_total_ms"])
kv_arr       = np.array(results["kv_total_ms"])
speedup_arr  = np.array(results["speedup"])

# ── Panel 1: Absolute wall-clock time ──────────────────────────────────────
ax = axes[0]
ax.plot(seq_arr, naive_arr, "o-", color="#E05C5C", linewidth=2.5,
        markersize=7, label="Naive (recomputes everything)")
ax.plot(seq_arr, kv_arr,   "o-", color="#5C9CE0", linewidth=2.5,
        markersize=7, label="KV Cache (incremental)")
ax.set_xlabel("Sequence Length (tokens)", fontsize=11)
ax.set_ylabel("Total Time (ms)", fontsize=11)
ax.set_title("Wall-Clock Time\nto Generate Full Sequence", fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
# Annotate the gap at the longest sequence
ax.annotate("", xy=(seq_arr[-1], naive_arr[-1]),
            xytext=(seq_arr[-1], kv_arr[-1]),
            arrowprops=dict(arrowstyle="<->", color="black", lw=1.5))
ax.text(seq_arr[-1] * 1.02, (naive_arr[-1] + kv_arr[-1]) / 2,
        f"{speedup_arr[-1]:.1f}×\nfaster", fontsize=9, va="center")

# ── Panel 2: Speedup ratio ─────────────────────────────────────────────────
ax = axes[1]
ax.plot(seq_arr, speedup_arr, "o-", color="#5CB85C", linewidth=2.5, markersize=7)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1.2, label="No speedup (1×)")
ax.fill_between(seq_arr, 1.0, speedup_arr, alpha=0.15, color="#5CB85C")
ax.set_xlabel("Sequence Length (tokens)", fontsize=11)
ax.set_ylabel("Speedup (×)", fontsize=11)
ax.set_title("Speedup of KV Cache\nover Naive", fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Panel 3: Log-log plot to reveal scaling exponent ──────────────────────
ax = axes[2]
ax.loglog(seq_arr, naive_arr, "o-", color="#E05C5C", linewidth=2.5,
          markersize=7, label="Naive")
ax.loglog(seq_arr, kv_arr,   "o-", color="#5C9CE0", linewidth=2.5,
          markersize=7, label="KV Cache")

# Fit lines to show the scaling exponents
if len(seq_arr) >= 3:
    naive_coef = np.polyfit(np.log(seq_arr), np.log(naive_arr), 1)
    kv_coef    = np.polyfit(np.log(seq_arr), np.log(kv_arr),   1)
    ax.set_title(f"Log-Log Scaling\n"
                 f"Naive ∝ n^{naive_coef[0]:.2f}, KV ∝ n^{kv_coef[0]:.2f}", fontsize=11)
else:
    ax.set_title("Log-Log Scaling", fontsize=11)

ax.set_xlabel("Sequence Length (tokens) — log scale", fontsize=11)
ax.set_ylabel("Total Time (ms) — log scale", fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which="both")

plt.tight_layout()
plt.savefig("kv_cache_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Three panels decoded:")
print("   Left  : Absolute time — the KV cache gap widens dramatically with sequence length.")
print("   Centre: Speedup ratio — how many times faster KV cache is at each length.")
print("   Right : Log-log plot — the slope reveals the scaling exponent (should be ~2 vs ~1).")

### What Does the Log-Log Plot Tell Us?

In a log-log plot, a power law $T \propto n^\alpha$ appears as a straight line with slope $\alpha$. If our theory is right:
- **Naive** should have slope $\approx 2$ (quadratic in $n$)
- **KV Cache** should have slope $\approx 1$ (linear in $n$)

The measured exponents from your data confirm exactly this. The KV cache genuinely converts an $O(n^2)$ computation into $O(n)$.

## 8. 🎯 Final Output — The Complete Story in One Visualisation

Let us produce a publication-quality figure that tells the complete KV cache story: what is cached, why it saves work, and what the cost looks like on the roofline.

In [ ]:
# 📊 The "hero's journey" — a four-panel narrative visualisation

fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor("#0F1117")

colours = {
    "naive"   : "#E05C5C",
    "kv"      : "#5CE08C",
    "query"   : "#FFD700",
    "key"     : "#87CEEB",
    "value"   : "#DDA0DD",
    "cache"   : "#98FB98",
    "bg_panel": "#1A1D27",
    "text"    : "#ECECEC",
    "subtext" : "#9CA3AF",
}

# ── Panel A: Token-by-token breakdown of what KV cache stores ─────────────
ax_a = fig.add_subplot(2, 2, 1)
ax_a.set_facecolor(colours["bg_panel"])

n_tokens = 6
token_labels = [f"tok {i+1}" for i in range(n_tokens)]
bar_width = 0.25
x = np.arange(n_tokens)

# Show Q, K, V components per token
bars_q = ax_a.bar(x - bar_width, [1]*n_tokens, bar_width,
                   label="Query (Q) — NEW each step",
                   color=colours["query"], alpha=0.9, edgecolor="white", linewidth=0.5)
bars_k = ax_a.bar(x,              [1]*n_tokens, bar_width,
                   label="Key (K) — CACHED ✅",
                   color=colours["key"], alpha=0.9, edgecolor="white", linewidth=0.5)
bars_v = ax_a.bar(x + bar_width, [1]*n_tokens, bar_width,
                   label="Value (V) — CACHED ✅",
                   color=colours["value"], alpha=0.9, edgecolor="white", linewidth=0.5)

# Dim the Q bars (they change so we can not cache them)
for b in bars_q:
    b.set_alpha(0.4)
    b.set_hatch("//")

ax_a.set_xticks(x)
ax_a.set_xticklabels(token_labels, color=colours["text"], fontsize=9)
ax_a.set_yticks([])
ax_a.set_title("What Does the Cache Store?", color=colours["text"], fontsize=11, pad=10)
ax_a.set_xlabel("Token Position", color=colours["subtext"], fontsize=9)
ax_a.legend(loc="upper right", fontsize=7.5,
            facecolor=colours["bg_panel"], edgecolor="gray",
            labelcolor=colours["text"])
ax_a.spines[:].set_color("#333")
ax_a.tick_params(colors=colours["subtext"])

note = "Queries change every step.\nKeys & Values are permanent once computed."
ax_a.text(0.02, 0.05, note, transform=ax_a.transAxes,
           fontsize=8, color=colours["subtext"], style="italic",
           bbox=dict(facecolor="#111", alpha=0.6, edgecolor="none", boxstyle="round,pad=0.3"))

# ── Panel B: Work done per step (naive vs KV cache) ───────────────────────
ax_b = fig.add_subplot(2, 2, 2)
ax_b.set_facecolor(colours["bg_panel"])

steps = np.arange(1, 25)
naive_work = steps ** 2       # O(t^2)
kv_work    = steps            # O(t)

ax_b.fill_between(steps, naive_work, alpha=0.3, color=colours["naive"])
ax_b.fill_between(steps, kv_work,    alpha=0.3, color=colours["kv"])
ax_b.plot(steps, naive_work, "-", color=colours["naive"],   linewidth=2.5, label="Naive: O(t²)")
ax_b.plot(steps, kv_work,    "-", color=colours["kv"],      linewidth=2.5, label="KV Cache: O(t)")

ax_b.set_xlabel("Decoding Step t", color=colours["subtext"], fontsize=9)
ax_b.set_ylabel("Attention FLOPs (relative)", color=colours["subtext"], fontsize=9)
ax_b.set_title("Work per Decoding Step", color=colours["text"], fontsize=11, pad=10)
ax_b.legend(fontsize=9, facecolor=colours["bg_panel"],
            edgecolor="gray", labelcolor=colours["text"])
ax_b.spines[:].set_color("#333")
ax_b.tick_params(colors=colours["subtext"])
ax_b.grid(True, alpha=0.15)

savings_pct = 100 * (1 - kv_work[-1] / naive_work[-1])
ax_b.text(0.55, 0.25, f"Saves {savings_pct:.0f}%\nof attention FLOPs\nat step {steps[-1]}",
           transform=ax_b.transAxes, fontsize=9, color=colours["text"],
           bbox=dict(facecolor="#111", alpha=0.7, edgecolor="none", boxstyle="round,pad=0.4"))

# ── Panel C: KV Cache memory footprint ────────────────────────────────────
ax_c = fig.add_subplot(2, 2, 3)
ax_c.set_facecolor(colours["bg_panel"])

seq_lengths_mem = np.array([512, 1024, 2048, 4096, 8192])
num_layers = 32
num_heads  = 32
d_head_mem = 128
bytes_per_param = 2   # float16

# KV cache size = 2 (K+V) × layers × heads × seq_len × d_head × bytes
cache_bytes = 2 * num_layers * num_heads * seq_lengths_mem * d_head_mem * bytes_per_param
cache_gb    = cache_bytes / 1e9

bars = ax_c.bar(range(len(seq_lengths_mem)), cache_gb,
                color=[colours["kv"] if g < 6 else colours["naive"] for g in cache_gb],
                edgecolor="white", linewidth=0.5, alpha=0.85)

ax_c.axhline(6,  color="#FFD700", linestyle="--", linewidth=1.5, label="~6 GB (T4 free VRAM)")
ax_c.axhline(16, color="#E05C5C", linestyle="--", linewidth=1.5, label="~16 GB (A100 40GB budget)")

ax_c.set_xticks(range(len(seq_lengths_mem)))
ax_c.set_xticklabels([f"{s//1024}K" if s >= 1024 else str(s)
                       for s in seq_lengths_mem],
                      color=colours["text"], fontsize=9)
ax_c.set_ylabel("KV Cache Size (GB)", color=colours["subtext"], fontsize=9)
ax_c.set_title(f"KV Cache Memory\n(GPT-style: {num_layers}L, {num_heads}H, d_h={d_head_mem}, fp16)",
               color=colours["text"], fontsize=11, pad=10)
ax_c.legend(fontsize=8, facecolor=colours["bg_panel"],
            edgecolor="gray", labelcolor=colours["text"])
ax_c.spines[:].set_color("#333")
ax_c.tick_params(colors=colours["subtext"])
ax_c.grid(True, alpha=0.15, axis="y")

# ── Panel D: Roofline sketch ───────────────────────────────────────────────
ax_d = fig.add_subplot(2, 2, 4)
ax_d.set_facecolor(colours["bg_panel"])

ai  = np.logspace(-1, 2, 300)       # arithmetic intensity (FLOPs/byte)
bw  = 1.5e12                         # ~1.5 TB/s (H100-ish bandwidth)
fp  = 80e12                          # ~80 TFLOPs/s peak
ridge = fp / bw                      # ridge point

roofline = np.minimum(ai * bw, fp * np.ones_like(ai)) / 1e12  # TFLOPs

ax_d.plot(ai, roofline, color="white", linewidth=2.5, zorder=5)
ax_d.fill_between(ai, roofline, alpha=0.12, color="white")

# Label the two regimes
ax_d.text(0.15, 0.25, "Memory-\nBound", color=colours["subtext"],
           fontsize=10, transform=ax_d.transAxes, ha="center")
ax_d.text(0.75, 0.75, "Compute-\nBound", color=colours["subtext"],
           fontsize=10, transform=ax_d.transAxes, ha="center")

# Mark where LLM inference (with KV cache) lives
ai_llm = 0.3
perf_llm = min(ai_llm * bw, fp) / 1e12
ax_d.scatter([ai_llm], [perf_llm], s=180, color=colours["kv"],
              zorder=10, edgecolors="white", linewidth=1.5)
ax_d.annotate("KV Cache Inference\n(memory-bound ⚠️)",
               xy=(ai_llm, perf_llm),
               xytext=(ai_llm * 4, perf_llm * 1.5),
               arrowprops=dict(arrowstyle="->", color=colours["kv"], lw=1.5),
               color=colours["kv"], fontsize=8.5,
               bbox=dict(facecolor="#111", alpha=0.7, edgecolor="none", boxstyle="round,pad=0.3"))

ax_d.axvline(ridge, color="#FFD700", linestyle=":", linewidth=1.2, alpha=0.5)
ax_d.text(ridge * 1.1, roofline.max() * 0.95,
           f"Ridge point\n({ridge:.0f} FLOPs/byte)",
           color="#FFD700", fontsize=7.5)

ax_d.set_xscale("log")
ax_d.set_xlabel("Arithmetic Intensity (FLOPs / byte)", color=colours["subtext"], fontsize=9)
ax_d.set_ylabel("Achieved Performance (TFLOPs/s)", color=colours["subtext"], fontsize=9)
ax_d.set_title("Roofline Model\n(KV cache slides us into memory-bound territory)",
               color=colours["text"], fontsize=11, pad=10)
ax_d.spines[:].set_color("#333")
ax_d.tick_params(colors=colours["subtext"])
ax_d.grid(True, alpha=0.15, which="both")

# ── Overall title ──────────────────────────────────────────────────────────
fig.text(0.5, 0.97,
         "The KV Cache: Hero of Inference — and its Hidden Cost",
         ha="center", va="top", fontsize=15, fontweight="bold",
         color=colours["text"])

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("kv_cache_full_story.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

print("\n🎯 Saved 'kv_cache_full_story.png' — a complete four-panel story of the KV cache.")
print("   Screenshot this and share it — you have earned it! 🏆")

## 9. Reflection and Next Steps

### What We Built and Learned

Let us take stock of everything we covered in this notebook:

1. **The Problem** — Naive autoregressive generation recomputes the full attention matrix ($O(n^2)$) at every single decoding step, leading to $O(n^3)$ total cost for an $n$-token sequence.

2. **The Key Observation** — A token's key and value vectors depend *only* on that token's embedding and the fixed model weights. They are immutable once computed. Only the query changes at each step.

3. **The KV Cache** — By storing keys and values in a growing buffer, each decoding step needs only:
   - Compute $q_t$, $k_t$, $v_t$ for the **one new token**.
   - Append $k_t$, $v_t$ to the cache.
   - Run a $(1, t)$ dot product — $O(t)$ instead of $O(t^2)$.

4. **The Measured Speedup** — We confirmed empirically that the KV cache produces *identical outputs* to naive attention but runs dramatically faster, with the gap widening super-linearly as sequence length grows.

5. **The Hidden Cost (Roofline)** — The KV cache trades arithmetic work for memory bandwidth. At each step we stream the entire $(t \times d_h)$ cache through the memory bus, but perform only $O(t \cdot d_h)$ floating-point operations. This gives a very low arithmetic intensity, placing LLM inference firmly in the **memory-bound** regime of the roofline — the GPU is often waiting for data, not running out of compute.

### 🤔 Reflection Questions

Think carefully about each of these before moving on:

1. **The cache never shrinks.** In our implementation, once we have cached 1,000 tokens of keys and values, every subsequent step must stream all 1,000 rows through the memory bus. How does this interact with the GPU's memory bandwidth limit? Would a longer context always be slower per step?

2. **Multi-head attention** runs $H$ attention heads in parallel. Each head has its own $K$ and $V$ matrices. If we have 32 heads and our sequence grows to 4,096 tokens, how does the total cache size scale? (Hint: work it out in bytes for a typical 7B-parameter model with fp16 weights.)

3. **The roofline tells us** that the KV cache is memory-bound, not compute-bound. What would you need to change about the computation to shift it rightward on the roofline (toward higher arithmetic intensity)?

4. **Correctness check** — we verified that the KV cache and naive attention agree. Can you think of a scenario where they might *not* agree — for example, if we used a positional encoding scheme that depends on the absolute length of the full sequence?

5. **The elephant in the room** — the KV cache must fit in VRAM alongside the model weights. For a 70B-parameter model with fp16 weights (~140 GB) and a 32K-token context, estimate the KV cache size. Does this fit on a single 80 GB A100?

### 🏆 Optional Challenges

Try these if you want to go deeper:

- **Challenge 1 (Medium):** Extend the `KVCachedAttention` class to support **multi-head attention** — run $H$ heads in parallel, each with its own cache, and concatenate their outputs.

- **Challenge 2 (Medium):** Implement a **sliding window cache** that only keeps the most recent $W$ tokens in the cache (dropping the oldest entry when the cache exceeds size $W$). Measure how this affects output quality vs. memory usage.

- **Challenge 3 (Hard):** Implement **Grouped Query Attention (GQA)**, where $H$ query heads share a smaller number of $K$ and $V$ head pairs (e.g., 8 KV heads for 32 query heads). Measure the reduction in cache memory footprint.

- **Challenge 4 (Research-level):** Read the [PagedAttention paper (vLLM)](https://arxiv.org/abs/2309.06180) and sketch out how it solves the memory fragmentation problem that arises when multiple sequences of different lengths share a single GPU's KV cache.

In [ ]:
# 🎉 Final summary printout

print("=" * 60)
print("  Section 3 Complete: The KV Cache — The Hero We Needed")
print("=" * 60)
print()
print("  ✅ Implemented NaiveAutoregressiveAttention   O(n²) per step")
print("  ✅ Implemented KVCachedAttention              O(n)  per step")
print("  ✅ Verified numerical equivalence             outputs match ✓")
print("  ✅ Benchmarked across sequence lengths        speedup measured")
print("  ✅ Visualised the roofline model              memory-bound ⚠️")
print()
print("  Key Takeaway:")
print("  The KV cache is brilliant — it eliminates redundant computation.")
print("  But it does not make inference free. It shifts the bottleneck")
print("  from arithmetic (compute-bound) to data movement (memory-bound).")
print()
print("  Next up → Section 4: Memory is the New Bottleneck")
print("  We will quantify exactly how fast we can stream the KV cache")
print("  and derive the memory-bandwidth-limited generation throughput.")
print("=" * 60)

---

*This notebook is part of the Vizuara series on Efficient LLM Inference.*
*Built with 💙 for curious minds who want to understand, not just use.*